<div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 30px; border-radius: 8px; margin-bottom: 25px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);">
    <h1 style="color: #f8fafc; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 700; margin: 0; font-size: 2.2em; letter-spacing: -0.5px;"> Procesamiento datos deberes PISA 🌍
</h1>
    <p style="color: #94a3b8; font-family: 'Segoe UI', sans-serif; margin: 8px 0 0 0; font-size: 1.1em;">Obtención de los datos para el analisis de al eficiencia del estudio (deberes) </p>
    
</div>

# 📋 Descripción  

Este cuaderno registra la obtención de datos para el estudio de la eficiencia del estudio de los alumnos, partiremos de la submuestra de colegios generada en el análisis global incorporando los datos necesarior para este apartado.

Seleccion de variables:

**Estudiantes (`STU_QQQ_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `ST296Q01JA` | Tiempo diario dedicado a los deberes de Matemáticas. |
| `ST296Q04JA` | Tiempo total acumulado diario en deberes de todas las asignaturas. |
| `STUDYHMW` | Índice derivado de frecuencia semanal de estudio antes/después de clase. |
| `ST307Q07JA` | Perseverancia: *"Dejo de hacer los deberes si son demasiado largos"*. |
| `ST309Q05JA` | Autorregulación: *"Reviso cuidadosamente los deberes antes de entregarlos"*. |
| `ESCS` | Índice socioeconómico individual (control obligatorio). |
| `PERSEVAGR` / `MATHPERS` | Índices estandarizados de perseverancia general y esfuerzo en matemáticas. |
| `W_FSTUWT` / `SENWT` / `STRATUM`  (Ajustados)| Pesos de muestreo (final, senado y estrato) para el diseño muestral. |

 **Escuelas (`SCH_QQQ_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `SC212Q01JA` | Disponibilidad de salas donde el alumnado puede hacer los deberes. |
| `SC212Q02JA` | Disponibilidad de personal del centro que ayuda con los deberes. |
| `SC212Q03JA` | Disponibilidad de tutoría de deberes entre iguales (*peer-to-peer*). |

**Tiempos (`STU_TIM_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `ST296_TT`, `ST307_TT`, `ST309_TT` | Tiempo de respuesta asociado a los bloques de preguntas sobre deberes, perseverancia y autorregulación. |

**Cognitivo (`STU_COG_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `PV1MATH` a `PV10MATH` | Los 10 Valores Plausibles de matemáticas (**variables objetivo / target**). |

# Preparación de los datos

Se carga la submuestra de escuelas creada en el cuaderno global y se prepara la lectura por bloques del fichero de estudiantes. La tabla inicial contendrá las variables de deberes y los identificadores necesarios para filtrar y relacionar los datos.

In [41]:
import gc
import os
import pickle
import sys
from pathlib import Path
import missingno as msno

#Se indica la ruta del workspace principal (Dos niveles atras)
sys.path.append(os.path.abspath('../../'))

#Se indica la comprobacion de cambio de librerias externas
%load_ext autoreload
%autoreload 2



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:

import numpy as np
import pandas as pd
import pyreadstat
from src.utils_pisa import *

In [8]:

ruta_datos_raw = Path('../../data/raw')
ruta_datos_intermedios = Path('../../data/intermediate')
ruta_datos_estudiante = ruta_datos_raw / 'CY08MSP_STU_QQQ.SAS7BDAT'
ruta_datos_escuelas = ruta_datos_raw / 'CY08MSP_SCH_QQQ.SAS7BDAT'
ruta_datos_tiempos = ruta_datos_raw / 'CY08MSP_STU_TIM.SAS7BDAT'
ruta_submuestra = ruta_datos_intermedios / 'escuelas_seleccionadas.pkl'
ruta_pesos_ajustados= ruta_datos_intermedios / 'pesos_senado_alumno_mapeados.pkl'


In [9]:
establecer_semilla()

✅ Semillas del proyecto establecidas con éxito (seed=42).


In [35]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Lectura de estudiantes por bloques

Se conservan las variables de interés del cuestionario de estudiantes y las claves que permiten filtrar las escuelas seleccionadas.

In [10]:
#Cargamos el set de datos donde se indica el ID de las escuelas seleccionadas y los factoresd e inflacion
escuelas_seleccionadas_set, factores_dict = cargar_pickle(ruta_submuestra)

#Carrgamos los pesos de alumnos y senado ajustados
df_pesos_ajustados = cargar_pickle(ruta_pesos_ajustados)



 Objetos cargados correctamente desde: ..\..\data\intermediate\escuelas_seleccionadas.pkl
 Objetos cargados correctamente desde: ..\..\data\intermediate\pesos_senado_alumno_mapeados.pkl


In [11]:
list(escuelas_seleccionadas_set)[:5]

[('BRA', 'BRA14', np.float64(7600053.0)),
 ('SLV', 'SLV16', np.float64(22200183.0)),
 ('BRA', 'BRA14', np.float64(7600174.0)),
 ('NOR', 'NOR97', np.float64(57800066.0)),
 ('KSV', 'KSV01', np.float64(38300016.0))]

In [12]:
df_pesos_ajustados.head()

,CNT,CNTSCHID,CNTSTUID,W_FSTUWT_adj,SENWT_adj
0,ALB,800282.0,800001.0,15.79370,2.837345
1,ALB,800242.0,800003.0,39.19300,7.041040
2,ALB,800172.0,800007.0,21.29730,3.826069
3,ALB,800040.0,800015.0,22.06520,3.964023
4,ALB,800123.0,800024.0,20.61752,3.703947


In [13]:
#Establecemos las columnas de interes para cada uno de los dataframes que vamos a procesar

#STU_QQQ.SAS7BDAT
columnas_estudiantes = [
    'CNT', 'STRATUM', 'CNTSCHID', 'CNTSTUID',
    'ST296Q01JA', 'ST296Q04JA', 'STUDYHMW',
    'ST307Q07JA', 'ST309Q05JA', 'ESCS',
    'PERSEVAGR', 'MATHPERS'
]

#SCH_QQQ.SAS7BDAT
columnas_escuelas = [
    'CNT', 'STRATUM', 'CNTSCHID',
    'SC212Q01JA', 'SC212Q02JA', 'SC212Q03JA'
]

#STU_TIM.SAS7BDAT
columnas_tiempos = [
    'CNT', 'CNTSTUID', 'ST296_TT', 'ST307_TT', 'ST309_TT'
]


In [14]:
#Reader para el archivo de estudiantes en chunks (el mas grande de todos) 
chunk_size = 10_000
reader = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sas7bdat,
    str(ruta_datos_estudiante),
    chunksize=chunk_size,
    usecols=columnas_estudiantes
)

In [15]:

chunks_filtrados = []

for chunk, _ in reader:
    #Creamos la clave temporal en forma de tupla en cada chunk
    chunk['clave_colegio_temp'] = list(zip(
        chunk['CNT'],
        chunk['STRATUM'],
        chunk['CNTSCHID']
    ))

    #De cada chunk nos quedamos solo con las escuelas presentes en el set
    chunk_filtrado = chunk.loc[
        chunk['clave_colegio_temp'].isin(escuelas_seleccionadas_set),
        columnas_estudiantes
    ].copy()

    if not chunk_filtrado.empty:
        chunks_filtrados.append(chunk_filtrado) #Añadimos el chunk filtrado resultante a la lista de chunks

if not chunks_filtrados:
    raise ValueError('La lectura no encontró estudiantes de las escuelas seleccionadas.')

df_estudiantes_submuestra = pd.concat(chunks_filtrados, ignore_index=True)
del chunks_filtrados
gc.collect()

df_estudiantes_submuestra.shape

(146500, 12)

In [16]:
df_estudiantes_submuestra.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 146500 entries, 0 to 146499
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   CNT         146500 non-null  str    
 1   STRATUM     146500 non-null  str    
 2   CNTSCHID    146500 non-null  float64
 3   CNTSTUID    146500 non-null  float64
 4   ST296Q01JA  129738 non-null  float64
 5   ST296Q04JA  129071 non-null  float64
 6   STUDYHMW    137226 non-null  float64
 7   ST307Q07JA  61702 non-null   float64
 8   ST309Q05JA  66206 non-null   float64
 9   ESCS        140451 non-null  float64
 10  PERSEVAGR   83349 non-null   float64
 11  MATHPERS    119631 non-null  float64
dtypes: float64(10), str(2)
memory usage: 14.6 MB


## Lectura de escuelas & Tiempos

Se cargan las variables de apoyo escolar disponibles en el fichero de escuelas y los tiempos de respuesta de los bloques relacionados con deberes. En ambos casos se filtran los registros de la submuestra.

### Escuelas

In [17]:
# El fichero de escuelas es reducido; se carga solo con las columnas necesarias.
df_escuelas = pyreadstat.read_sas7bdat(
    str(ruta_datos_escuelas),
    usecols=columnas_escuelas
)[0] #Nos quedamos solo con los datos, no los metadatos


In [18]:
#Realizamos el mismo procedimento pero sin leer en chunks 

df_escuelas['clave_colegio_temp'] = list(zip( #Generamos una tupla que contiene la clave a buscar
    df_escuelas['CNT'],
    df_escuelas['STRATUM'],
    df_escuelas['CNTSCHID']
))

df_escuelas_submuestra = df_escuelas.loc[ #Buscamos en el set la identificacion de los paises de la submuestra las correspondencias
    df_escuelas['clave_colegio_temp'].isin(escuelas_seleccionadas_set),
    columnas_escuelas
].copy()
del df_escuelas

if df_escuelas_submuestra.empty:
    raise ValueError('La lectura no encontró escuelas de la submuestra en el fichero de escuelas.')

df_escuelas_submuestra.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 5113 entries, 3 to 21624
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CNT         5113 non-null   str    
 1   STRATUM     5113 non-null   str    
 2   CNTSCHID    5113 non-null   float64
 3   SC212Q01JA  4622 non-null   float64
 4   SC212Q02JA  4627 non-null   float64
 5   SC212Q03JA  4629 non-null   float64
dtypes: float64(4), str(2)
memory usage: 320.8 KB


### Tiempos

Realizamos la lectura por bloques tambien

In [19]:
reader_tiempos = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sas7bdat,
    str(ruta_datos_tiempos),
    chunksize=chunk_size,
    usecols=columnas_tiempos
)

In [20]:
chunks_tiempos_filtrados = []

#Generamos las claves
claves_estudiantes = set(zip(
    df_estudiantes_submuestra['CNT'],
    df_estudiantes_submuestra['CNTSTUID']
))

for chunk_tiempos, _ in reader_tiempos:
    claves_chunk = list(zip(chunk_tiempos['CNT'], chunk_tiempos['CNTSTUID']))
    chunk_tiempos_filtrado = chunk_tiempos.loc[
        pd.Series(claves_chunk, index=chunk_tiempos.index).isin(claves_estudiantes),
        columnas_tiempos
    ].copy()

    if not chunk_tiempos_filtrado.empty:
        chunks_tiempos_filtrados.append(chunk_tiempos_filtrado)

if not chunks_tiempos_filtrados:
    raise ValueError('La lectura no encontró registros de tiempo para los estudiantes de la submuestra.')



In [21]:
df_tiempos_submuestra = pd.concat(chunks_tiempos_filtrados, ignore_index=True)
df_tiempos_submuestra.info()

<class 'pandas.DataFrame'>
RangeIndex: 146500 entries, 0 to 146499
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   CNT       146500 non-null  str    
 1   CNTSTUID  146500 non-null  float64
 2   ST296_TT  135516 non-null  float64
 3   ST307_TT  131728 non-null  float64
 4   ST309_TT  130351 non-null  float64
dtypes: float64(4), str(1)
memory usage: 6.0 MB


In [22]:
#Limpiamos la memorias
del chunks_tiempos_filtrados, claves_chunk
gc.collect()

0

In [23]:

df_analisis_deberes = (
    df_estudiantes_submuestra
    .merge(
        df_escuelas_submuestra,
        on=['CNT', 'STRATUM', 'CNTSCHID'],
        how='left',
        validate='many_to_one'    # Escuelas (Muchos estudiantes -> Una escuela)
   
    )
    .merge(
        df_tiempos_submuestra,
        on=['CNT', 'CNTSTUID'],
        how='left',
        validate='one_to_one' # Tiempos (Un estudiante -> Un registro de tiempos)
    )
    .merge(
        df_pesos_ajustados,
        on=['CNT', 'CNTSCHID', 'CNTSTUID'],
        how='left',
        validate='one_to_one'     # Pesos Ponderados (Un estudiante -> Un conjunto de pesos)
    )
)



In [24]:
df_analisis_deberes.info()

<class 'pandas.DataFrame'>
RangeIndex: 146500 entries, 0 to 146499
Data columns (total 20 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   CNT           146500 non-null  str    
 1   STRATUM       146500 non-null  str    
 2   CNTSCHID      146500 non-null  float64
 3   CNTSTUID      146500 non-null  float64
 4   ST296Q01JA    129738 non-null  float64
 5   ST296Q04JA    129071 non-null  float64
 6   STUDYHMW      137226 non-null  float64
 7   ST307Q07JA    61702 non-null   float64
 8   ST309Q05JA    66206 non-null   float64
 9   ESCS          140451 non-null  float64
 10  PERSEVAGR     83349 non-null   float64
 11  MATHPERS      119631 non-null  float64
 12  SC212Q01JA    134931 non-null  float64
 13  SC212Q02JA    135121 non-null  float64
 14  SC212Q03JA    135274 non-null  float64
 15  ST296_TT      135516 non-null  float64
 16  ST307_TT      131728 non-null  float64
 17  ST309_TT      130351 non-null  float64
 18  W_FSTUWT_adj  1

In [26]:
df_analisis_deberes.head(2)

,CNT,STRATUM,CNTSCHID,CNTSTUID,ST296Q01JA,ST296Q04JA,STUDYHMW,ST307Q07JA,ST309Q05JA,ESCS,PERSEVAGR,MATHPERS,SC212Q01JA,SC212Q02JA,SC212Q03JA,ST296_TT,ST307_TT,ST309_TT,W_FSTUWT_adj,SENWT_adj
0,ALB,ALB03,800282.0,800001.0,3.0,6.0,10.0,NaN,5.0,1.1112,NaN,-0.0085,1.0,1.0,1.0,23779.0,177.0,28351.0,15.7937,2.837345
1,ALB,ALB01,800242.0,800003.0,2.0,2.0,0.0,NaN,NaN,-0.1867,NaN,NaN,2.0,2.0,1.0,16409.0,573.0,554.0,39.1930,7.041040


In [27]:
#Guardamos eldataframe sin procesaro en formato parquet
ruta_df_analisis_deberes = ruta_datos_intermedios/'pisa_df_deberes_no_procesado.parquet'



In [28]:
df_analisis_deberes.to_parquet(ruta_df_analisis_deberes,engine='pyarrow', index=False)

In [37]:
df_analisis_deberes = pd.read_parquet(ruta_df_analisis_deberes)

# EDA

In [38]:
#Trasnformamos los ID que aparecen como floar en STR
#TODO ver que hacer para que no se conviertan en float
columnas_ids = ['CNTSCHID', 'CNTSCHID']
for col in columnas_ids:

    df_analisis_deberes[col] = df_analisis_deberes[col].astype('Int64').astype(str)

In [39]:
# Convertimos las categorías de duración a horas y damos nombres descriptivos a las columnas.
# Según el diccionario PISA: 1 (<30m), 2 (30m-1h), 3 (1-2h), 4 (2-3h), 5 (3-4h), 6 (>4h).
mapeo_horas = {
    1: 0.25,
    2: 0.75,
    3: 1.50,
    4: 2.50,
    5: 3.50,
    6: 4.50
}

columnas_horas = ['ST296Q01JA', 'ST296Q04JA']
for columna in columnas_horas:
    df_analisis_deberes[columna] = df_analisis_deberes[columna].map(mapeo_horas)

renombrado_columnas = {

    'ST296Q01JA': 'Horas_deberes_mates', # Categórica original
    'ST296Q04JA': 'Horas_deberes_totales',     # Categórica original (La clave para mapear a horas)
    'STUDYHMW': 'Indice_Frecuencia_Estudio',
    'ST307Q07JA': 'Abandono_Por_Fatiga',     # "Dejo los deberes si son largos"
    'ST309Q05JA': 'Revision_Deberes',        # "Reviso antes de entregar"
    'PERSEVAGR': 'Perseverancia_General',
    'MATHPERS': 'Perseverancia_Matematicas',
    'SC212Q01JA': 'salas_estudio_disponible',
    'SC212Q02JA': 'personal_apoyo_disponible',
    'SC212Q03JA': 'tutorias_disponible',
    'ST296_TT': 'Tiempo_Respuesta_Q_Deberes',
    'ST307_TT': 'Tiempo_Respuesta_Q_Fatiga',
    'ST309_TT': 'Tiempo_Respuesta_Q_Revision'
}

df_analisis_deberes = df_analisis_deberes.rename(columns=renombrado_columnas)
df_analisis_deberes.head()

,CNT,STRATUM,CNTSCHID,CNTSTUID,Horas_deberes_mates,Horas_deberes_totales,Indice_Frecuencia_Estudio,Abandono_Por_Fatiga,Revision_Deberes,ESCS,Perseverancia_General,Perseverancia_Matematicas,salas_estudio_disponible,personal_apoyo_disponible,tutorias_disponible,Tiempo_Respuesta_Q_Deberes,Tiempo_Respuesta_Q_Fatiga,Tiempo_Respuesta_Q_Revision,W_FSTUWT_adj,SENWT_adj
0,ALB,ALB03,800282,800001.00,1.50,4.50,10.00,NaN,5.00,1.11,NaN,-0.01,1.00,1.00,1.00,23779.00,177.00,28351.00,15.79,2.84
1,ALB,ALB01,800242,800003.00,0.75,0.75,0.00,NaN,NaN,-0.19,NaN,NaN,2.00,2.00,1.00,16409.00,573.00,554.00,39.19,7.04
2,ALB,ALB04,800172,800007.00,1.50,2.50,10.00,5.00,5.00,1.09,NaN,2.34,1.00,1.00,2.00,18874.00,13632.00,19552.00,21.30,3.83
3,ALB,ALB03,800040,800015.00,NaN,NaN,NaN,NaN,NaN,-0.17,NaN,NaN,2.00,1.00,1.00,139.00,294.00,38.00,22.07,3.96
4,ALB,ALB06,800123,800024.00,1.50,4.50,10.00,NaN,NaN,-0.89,NaN,2.72,1.00,1.00,1.00,18427.00,30772.00,41648.00,20.62,3.70


In [40]:
#Calulamos el porcentaje de nulos del dataframe
pct_nulos_total = (df_analisis_deberes.isnull().sum() / len(df_analisis_deberes)) * 100
print(pct_nulos_total.sort_values(ascending=False).head(10))

Abandono_Por_Fatiga           57.88
Revision_Deberes              54.81
Perseverancia_General         43.11
Perseverancia_Matematicas     18.34
Horas_deberes_totales         11.90
Horas_deberes_mates           11.44
Tiempo_Respuesta_Q_Revision   11.02
Tiempo_Respuesta_Q_Fatiga     10.08
salas_estudio_disponible       7.90
personal_apoyo_disponible      7.77
dtype: float64


In [ ]:
msno.